In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

print("=" * 80)
print("РАЗДЕЛЬНЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ ТОЧНОСТИ: ФАГС vs IGS")
print("=" * 80)

# ============================================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================================

print("\nЗагрузка данных...")
df_fags_raw = pd.read_csv('/Users/sergeidolin/collaborative-service/fags_batch_report.csv')
df_igs_raw = pd.read_csv('/Users/sergeidolin/collaborative-service/igs_batch_report.csv')

print(f"ФАГС: {len(df_fags_raw)} записей, {df_fags_raw['station'].nunique()} станций")
print(f"IGS: {len(df_igs_raw)} записей, {df_igs_raw['station'].nunique()} станций")

# ============================================================================
# ДИАГНОСТИКА ДАННЫХ ФАГС
# ============================================================================
print("\n" + "=" * 80)
print("ДИАГНОСТИКА ДАННЫХ ФАГС")
print("=" * 80)

print(f"\nСтатусы измерений ФАГС:")
print(df_fags_raw['status'].value_counts())

print(f"\nКачество измерений ФАГС (q):")
print(df_fags_raw['q'].value_counts().sort_index())

print(f"\nДиапазон r3d_m в ФАГС:")
print(f"  min: {df_fags_raw['r3d_m'].min():.4f} м")
print(f"  max: {df_fags_raw['r3d_m'].max():.4f} м")
print(f"  mean: {df_fags_raw['r3d_m'].mean():.4f} м")

print(f"\nДиапазон nsat в ФАГС:")
print(f"  min: {df_fags_raw['nsat'].min()}")
print(f"  max: {df_fags_raw['nsat'].max()}")
print(f"  mean: {df_fags_raw['nsat'].mean():.1f}")

print(f"\nНулевые координаты в ФАГС:")
zero_lat = (df_fags_raw['lat'] == 0).sum()
zero_lon = (df_fags_raw['lon'] == 0).sum()
print(f"  lat=0: {zero_lat}, lon=0: {zero_lon}")

# ============================================================================
# ФУНКЦИЯ ДЛЯ ОБРАБОТКИ ДАННЫХ (С МЯГКОЙ ФИЛЬТРАЦИЕЙ)
# ============================================================================

def process_network_data(df, network_name, verbose=True):
    """Обработка данных с мягкой фильтрацией"""
    if len(df) == 0:
        return pd.DataFrame()
    
    df_processed = df.copy()
    
    if verbose:
        print(f"\n--- Обработка {network_name} ---")
        print(f"  Исходное количество записей: {len(df_processed)}")
    
    # Шаг 1: Оставляем только успешные измерения
    if 'status' in df_processed.columns:
        before = len(df_processed)
        df_processed = df_processed[df_processed['status'] == 'ok']
        if verbose:
            print(f"  После фильтрации status='ok': {len(df_processed)} (удалено {before - len(df_processed)})")
    
    # Шаг 2: Фильтрация по качеству (оставляем q=1,2,5,6 - все кроме 0)
    if 'q' in df_processed.columns and len(df_processed) > 0:
        before = len(df_processed)
        df_processed = df_processed[df_processed['q'] > 0]
        if verbose:
            print(f"  После фильтрации q>0: {len(df_processed)} (удалено {before - len(df_processed)})")
    
    # Шаг 3: Удаление явных выбросов по 3D ошибке (очень большие)
    if 'r3d_m' in df_processed.columns and len(df_processed) > 0:
        before = len(df_processed)
        # Более мягкое ограничение - удаляем только совсем аномальные (> 5 м)
        df_processed = df_processed[df_processed['r3d_m'] < 5.0]
        if verbose:
            print(f"  После удаления r3d_m > 5 м: {len(df_processed)} (удалено {before - len(df_processed)})")
    
    # Шаг 4: Удаление нулевых координат (если есть)
    if 'lat' in df_processed.columns and len(df_processed) > 0:
        before = len(df_processed)
        df_processed = df_processed[(df_processed['lat'] != 0) | (df_processed['lon'] != 0)]
        if verbose:
            print(f"  После удаления нулевых координат: {len(df_processed)} (удалено {before - len(df_processed)})")
    
    if len(df_processed) == 0:
        if verbose:
            print(f"  НЕТ ДАННЫХ после фильтрации!")
        return pd.DataFrame()
    
    # Расчет статистик по станциям
    stats_list = []
    
    for station in df_processed['station'].unique():
        station_data = df_processed[df_processed['station'] == station].copy()
        n_meas = len(station_data)
        
        # Минимальное количество измерений - 1 (для синглтонов используем только одно измерение)
        if n_meas < 1:
            continue
        
        # Для одной станции СКП = 0, используем только bias
        if n_meas == 1:
            skp_N = 0
            skp_E = 0
            skp_U = 0
            skp_3D = 0
        else:
            skp_N = station_data['dN_m'].std() * 1000 if len(station_data) > 1 else 0
            skp_E = station_data['dE_m'].std() * 1000 if len(station_data) > 1 else 0
            skp_U = station_data['dU_m'].std() * 1000 if len(station_data) > 1 else 0
            skp_3D = station_data['r3d_m'].std() * 1000 if len(station_data) > 1 else 0
        
        # Средние ошибки (в мм)
        bias_N = station_data['dN_m'].mean() * 1000
        bias_E = station_data['dE_m'].mean() * 1000
        bias_U = station_data['dU_m'].mean() * 1000
        bias_3D = station_data['r3d_m'].mean() * 1000
        
        # Статистика по спутникам
        mean_nsat = station_data['nsat'].mean()
        std_nsat = station_data['nsat'].std() if n_meas > 1 else 0
        
        stats_list.append({
            'station': station,
            'network': network_name,
            'n_meas': n_meas,
            'skp_N_mm': skp_N,
            'skp_E_mm': skp_E,
            'skp_U_mm': skp_U,
            'skp_3D_mm': skp_3D,
            'bias_N_mm': bias_N,
            'bias_E_mm': bias_E,
            'bias_U_mm': bias_U,
            'bias_3D_mm': bias_3D,
            'mean_nsat': mean_nsat,
            'std_nsat': std_nsat
        })
    
    df_stats = pd.DataFrame(stats_list)
    
    if verbose:
        print(f"  Рассчитано статистик для {len(df_stats)} станций")
    
    # Мягкое удаление выбросов - только явные (СКП_3D > 200 мм)
    if len(df_stats) > 0:
        before = len(df_stats)
        df_stats = df_stats[df_stats['skp_3D_mm'] <= 200]
        if verbose and before - len(df_stats) > 0:
            print(f"  Удалено станций с СКП_3D > 200 мм: {before - len(df_stats)}")
    
    return df_stats

# ============================================================================
# ОБРАБОТКА ОБОИХ СЕТЕЙ
# ============================================================================

print("\n" + "=" * 80)
print("ЭТАП 1: ФИЛЬТРАЦИЯ И РАСЧЕТ СТАТИСТИК")
print("=" * 80)

df_fags_stats = process_network_data(df_fags_raw, 'ФАГС', verbose=True)
df_igs_stats = process_network_data(df_igs_raw, 'IGS', verbose=True)

if len(df_fags_stats) == 0:
    print("\nОШИБКА: Нет данных для ФАГС! Проверьте структуру файла.")
    print("Отображаем первые 5 строк файла ФАГС для диагностики:")
    print(df_fags_raw.head())
    exit()

if len(df_igs_stats) == 0:
    print("\nПРЕДУПРЕЖДЕНИЕ: Нет данных для IGS!")
    df_igs_stats = pd.DataFrame(columns=df_fags_stats.columns)

print(f"\nИтоговое количество станций:")
print(f"  ФАГС: {len(df_fags_stats)} станций")
print(f"  IGS: {len(df_igs_stats)} станций")

# ============================================================================
# СРАВНИТЕЛЬНАЯ СТАТИСТИКА
# ============================================================================

print("\n" + "=" * 80)
print("ЭТАП 2: СРАВНИТЕЛЬНАЯ СТАТИСТИКА ФАГС vs IGS")
print("=" * 80)

def compute_summary(df, name):
    if len(df) == 0:
        return None
    
    return {
        'network': name,
        'count': len(df),
        'skp_N_mean': df['skp_N_mm'].mean(),
        'skp_N_std': df['skp_N_mm'].std(),
        'skp_E_mean': df['skp_E_mm'].mean(),
        'skp_E_std': df['skp_E_mm'].std(),
        'skp_U_mean': df['skp_U_mm'].mean(),
        'skp_U_std': df['skp_U_mm'].std(),
        'skp_3D_mean': df['skp_3D_mm'].mean(),
        'skp_3D_median': df['skp_3D_mm'].median(),
        'skp_3D_std': df['skp_3D_mm'].std(),
        'skp_3D_rms': np.sqrt(np.mean(df['skp_3D_mm']**2)),
        'skp_3D_min': df['skp_3D_mm'].min(),
        'skp_3D_max': df['skp_3D_mm'].max(),
        'bias_N_mean': df['bias_N_mm'].mean(),
        'bias_E_mean': df['bias_E_mm'].mean(),
        'bias_U_mean': df['bias_U_mm'].mean(),
        'mean_nsat_mean': df['mean_nsat'].mean(),
        'total_measurements': df['n_meas'].sum()
    }

summary_fags = compute_summary(df_fags_stats, 'ФАГС')
summary_igs = compute_summary(df_igs_stats, 'IGS') if len(df_igs_stats) > 0 else None

print("\nСРАВНИТЕЛЬНАЯ ТАБЛИЦА:")
print("-" * 100)
print(f"{'Параметр':<35} {'ФАГС':<30} {'IGS':<30}")
print("-" * 100)

print(f"{'Количество станций':<35} {summary_fags['count']:<30} {summary_igs['count'] if summary_igs else 'Нет данных':<30}")
print(f"{'Общее число измерений':<35} {summary_fags['total_measurements']:<30} {summary_igs['total_measurements'] if summary_igs else 'Нет данных':<30}")
print(f"{'':<35} {'':<30} {'':<30}")

print(f"{'СКП_N (ср.), мм':<35} {summary_fags['skp_N_mean']:.2f} ± {summary_fags['skp_N_std']:.2f}       {summary_igs['skp_N_mean']:.2f} ± {summary_igs['skp_N_std']:.2f}" if summary_igs else f"{'СКП_N (ср.), мм':<35} {summary_fags['skp_N_mean']:.2f} ± {summary_fags['skp_N_std']:.2f}")
print(f"{'СКП_E (ср.), мм':<35} {summary_fags['skp_E_mean']:.2f} ± {summary_fags['skp_E_std']:.2f}       {summary_igs['skp_E_mean']:.2f} ± {summary_igs['skp_E_std']:.2f}" if summary_igs else f"{'СКП_E (ср.), мм':<35} {summary_fags['skp_E_mean']:.2f} ± {summary_fags['skp_E_std']:.2f}")
print(f"{'СКП_U (ср.), мм':<35} {summary_fags['skp_U_mean']:.2f} ± {summary_fags['skp_U_std']:.2f}       {summary_igs['skp_U_mean']:.2f} ± {summary_igs['skp_U_std']:.2f}" if summary_igs else f"{'СКП_U (ср.), мм':<35} {summary_fags['skp_U_mean']:.2f} ± {summary_fags['skp_U_std']:.2f}")
print(f"{'СКП_3D (ср.), мм':<35} {summary_fags['skp_3D_mean']:.2f} ± {summary_fags['skp_3D_std']:.2f}       {summary_igs['skp_3D_mean']:.2f} ± {summary_igs['skp_3D_std']:.2f}" if summary_igs else f"{'СКП_3D (ср.), мм':<35} {summary_fags['skp_3D_mean']:.2f} ± {summary_fags['skp_3D_std']:.2f}")

print(f"{'СКП_3D (медиана), мм':<35} {summary_fags['skp_3D_median']:.2f}       {summary_igs['skp_3D_median']:.2f}" if summary_igs else f"{'СКП_3D (медиана), мм':<35} {summary_fags['skp_3D_median']:.2f}")
print(f"{'СКП_3D (СКО), мм':<35} {summary_fags['skp_3D_rms']:.2f}       {summary_igs['skp_3D_rms']:.2f}" if summary_igs else f"{'СКП_3D (СКО), мм':<35} {summary_fags['skp_3D_rms']:.2f}")
print(f"{'СКП_3D (min-max), мм':<35} {summary_fags['skp_3D_min']:.2f} - {summary_fags['skp_3D_max']:.2f}       {summary_igs['skp_3D_min']:.2f} - {summary_igs['skp_3D_max']:.2f}" if summary_igs else f"{'СКП_3D (min-max), мм':<35} {summary_fags['skp_3D_min']:.2f} - {summary_fags['skp_3D_max']:.2f}")

print(f"{'':<35} {'':<30} {'':<30}")
print(f"{'Сист. сдвиг dN, мм':<35} {summary_fags['bias_N_mean']:.2f}       {summary_igs['bias_N_mean']:.2f}" if summary_igs else f"{'Сист. сдвиг dN, мм':<35} {summary_fags['bias_N_mean']:.2f}")
print(f"{'Сист. сдвиг dE, мм':<35} {summary_fags['bias_E_mean']:.2f}       {summary_igs['bias_E_mean']:.2f}" if summary_igs else f"{'Сист. сдвиг dE, мм':<35} {summary_fags['bias_E_mean']:.2f}")
print(f"{'Сист. сдвиг dU, мм':<35} {summary_fags['bias_U_mean']:.2f}       {summary_igs['bias_U_mean']:.2f}" if summary_igs else f"{'Сист. сдвиг dU, мм':<35} {summary_fags['bias_U_mean']:.2f}")

print(f"{'':<35} {'':<30} {'':<30}")
print(f"{'Среднее NSAT':<35} {summary_fags['mean_nsat_mean']:.1f}       {summary_igs['mean_nsat_mean']:.1f}" if summary_igs else f"{'Среднее NSAT':<35} {summary_fags['mean_nsat_mean']:.1f}")

# ============================================================================
# КАТЕГОРИЗАЦИЯ
# ============================================================================

def get_category(skp_mm):
    if pd.isna(skp_mm) or skp_mm == 0:
        return "Неопределено"
    elif skp_mm < 5:
        return "Экстра-прецизионные (<5 мм)"
    elif skp_mm < 10:
        return "Прецизионные (5-10 мм)"
    elif skp_mm < 20:
        return "Высокоточные (10-20 мм)"
    elif skp_mm < 40:
        return "Средней точности (20-40 мм)"
    else:
        return "Пониженной точности (>40 мм)"

df_fags_stats['category'] = df_fags_stats['skp_3D_mm'].apply(get_category)
if len(df_igs_stats) > 0:
    df_igs_stats['category'] = df_igs_stats['skp_3D_mm'].apply(get_category)

print("\n" + "=" * 80)
print("ЭТАП 3: РАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ ТОЧНОСТИ")
print("=" * 80)

categories = ['Экстра-прецизионные (<5 мм)', 'Прецизионные (5-10 мм)', 
              'Высокоточные (10-20 мм)', 'Средней точности (20-40 мм)', 
              'Пониженной точности (>40 мм)', 'Неопределено']

print(f"\n{'Категория':<35} {'ФАГС':<15} {'IGS':<15}")
print("-" * 65)
for cat in categories:
    fags_count = len(df_fags_stats[df_fags_stats['category'] == cat])
    igs_count = len(df_igs_stats[df_igs_stats['category'] == cat]) if len(df_igs_stats) > 0 else 0
    if fags_count > 0 or igs_count > 0:
        print(f"{cat:<35} {fags_count:<15} {igs_count:<15}")

# ============================================================================
# СТАТИСТИЧЕСКИЙ ТЕСТ
# ============================================================================

if len(df_fags_stats) > 0 and len(df_igs_stats) > 0:
    print("\n" + "=" * 80)
    print("ЭТАП 4: СТАТИСТИЧЕСКАЯ ПРОВЕРКА (Манн-Уитни)")
    print("=" * 80)
    
    # Фильтруем нулевые СКП для теста
    fags_nonzero = df_fags_stats[df_fags_stats['skp_3D_mm'] > 0]['skp_3D_mm']
    igs_nonzero = df_igs_stats[df_igs_stats['skp_3D_mm'] > 0]['skp_3D_mm']
    
    if len(fags_nonzero) > 0 and len(igs_nonzero) > 0:
        stat, p_value = mannwhitneyu(fags_nonzero, igs_nonzero, alternative='two-sided')
        print(f"\nСравнение СКП_3D (ФАГС vs IGS):")
        print(f"  Статистика U: {stat:.2f}")
        print(f"  p-value: {p_value:.6f}")
        print(f"  Различие статистически значимо: {'ДА ✓' if p_value < 0.05 else 'НЕТ ✗'}")
        
        if p_value < 0.05:
            if summary_fags['skp_3D_median'] < summary_igs['skp_3D_median']:
                print(f"  Вывод: ФАГС показывает статистически значимо лучшую точность")
            else:
                print(f"  Вывод: IGS показывает статистически значимо лучшую точность")

# ============================================================================
# ПОСТРОЕНИЕ ГРАФИКОВ
# ============================================================================

print("\n" + "=" * 80)
print("ЭТАП 5: ПОСТРОЕНИЕ ГРАФИКОВ")
print("=" * 80)

fig = plt.figure(figsize=(18, 14))

# 1. Гистограммы
ax1 = fig.add_subplot(3, 3, 1)
fags_data = df_fags_stats[df_fags_stats['skp_3D_mm'] > 0]['skp_3D_mm']
ax1.hist(fags_data, bins=20, alpha=0.6, label='ФАГС', color='blue', edgecolor='black')
if len(df_igs_stats) > 0:
    igs_data = df_igs_stats[df_igs_stats['skp_3D_mm'] > 0]['skp_3D_mm']
    ax1.hist(igs_data, bins=20, alpha=0.6, label='IGS', color='red', edgecolor='black')
ax1.set_xlabel('СКП_3D (мм)')
ax1.set_ylabel('Количество станций')
ax1.set_title('Распределение СКП_3D')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Boxplot
ax2 = fig.add_subplot(3, 3, 2)
data_to_box = []
positions = []
colors_box = []
if len(fags_data) > 0:
    data_to_box.append(fags_data)
    positions.append(1)
    colors_box.append('blue')
if len(df_igs_stats) > 0 and len(igs_data) > 0:
    data_to_box.append(igs_data)
    positions.append(2)
    colors_box.append('red')
if data_to_box:
    bp = ax2.boxplot(data_to_box, positions=positions, widths=0.6, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax2.set_xticks(positions)
    ax2.set_xticklabels(['ФАГС'] if len(data_to_box) == 1 else ['ФАГС', 'IGS'])
ax2.set_ylabel('СКП_3D (мм)')
ax2.set_title('Сравнение СКП_3D')
ax2.grid(True, alpha=0.3, axis='y')

# 3. Категории
ax3 = fig.add_subplot(3, 3, 3)
categories_plot = [c for c in categories if c != 'Неопределено']
x = np.arange(len(categories_plot))
width = 0.35
fags_counts = [len(df_fags_stats[df_fags_stats['category'] == cat]) for cat in categories_plot]
ax3.bar(x - width/2, fags_counts, width, label='ФАГС', color='blue')
if len(df_igs_stats) > 0:
    igs_counts = [len(df_igs_stats[df_igs_stats['category'] == cat]) for cat in categories_plot]
    ax3.bar(x + width/2, igs_counts, width, label='IGS', color='red')
ax3.set_xticks(x)
ax3.set_xticklabels(categories_plot, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('Количество станций')
ax3.set_title('Распределение по категориям точности')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# 4. Систематические ошибки
ax4 = fig.add_subplot(3, 3, 4)
bias_components = ['dN', 'dE', 'dU']
x = np.arange(len(bias_components))
fags_biases = [df_fags_stats['bias_N_mm'].mean(), df_fags_stats['bias_E_mm'].mean(), df_fags_stats['bias_U_mm'].mean()]
ax4.axhline(0, color='black', linestyle='-', linewidth=0.8)
ax4.bar(x - width/2, fags_biases, width, label='ФАГС', color='blue')
if len(df_igs_stats) > 0:
    igs_biases = [df_igs_stats['bias_N_mm'].mean(), df_igs_stats['bias_E_mm'].mean(), df_igs_stats['bias_U_mm'].mean()]
    ax4.bar(x + width/2, igs_biases, width, label='IGS', color='red')
ax4.set_xticks(x)
ax4.set_xticklabels(bias_components)
ax4.set_ylabel('Систематическая ошибка (мм)')
ax4.set_title('Систематические ошибки')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 5. Зависимость от NSAT
ax5 = fig.add_subplot(3, 3, 5)
ax5.scatter(df_fags_stats['mean_nsat'], df_fags_stats['skp_3D_mm'], alpha=0.6, label='ФАГС', color='blue', s=50)
if len(df_igs_stats) > 0:
    ax5.scatter(df_igs_stats['mean_nsat'], df_igs_stats['skp_3D_mm'], alpha=0.6, label='IGS', color='red', s=50)
ax5.set_xlabel('Среднее NSAT')
ax5.set_ylabel('СКП_3D (мм)')
ax5.set_title('Зависимость СКП_3D от NSAT')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. Violin plot
ax6 = fig.add_subplot(3, 3, 6)
data_violin = []
positions = []
colors_violin = []
if len(fags_data) > 0:
    data_violin.append(fags_data)
    positions.append(1)
    colors_violin.append('blue')
if len(df_igs_stats) > 0 and len(igs_data) > 0:
    data_violin.append(igs_data)
    positions.append(2)
    colors_violin.append('red')
if data_violin:
    vp = ax6.violinplot(data_violin, positions=positions, showmeans=True, showmedians=True)
    for body, color in zip(vp['bodies'], colors_violin):
        body.set_facecolor(color)
        body.set_alpha(0.6)
    ax6.set_xticks(positions)
    ax6.set_xticklabels(['ФАГС'] if len(data_violin) == 1 else ['ФАГС', 'IGS'])
ax6.set_ylabel('СКП_3D (мм)')
ax6.set_title('Распределение СКП_3D (Violin plot)')
ax6.grid(True, alpha=0.3, axis='y')

# 7. Топ-10 ФАГС
ax7 = fig.add_subplot(3, 3, 7)
top_fags = df_fags_stats[df_fags_stats['skp_3D_mm'] > 0].nsmallest(10, 'skp_3D_mm')
if len(top_fags) > 0:
    y_pos = np.arange(len(top_fags))
    ax7.barh(y_pos, top_fags['skp_3D_mm'], color='blue', edgecolor='black')
    ax7.set_yticks(y_pos)
    ax7.set_yticklabels(top_fags['station'], fontsize=8)
    ax7.set_xlabel('СКП_3D (мм)')
    ax7.set_title('Топ-10 точных станций ФАГС')
    ax7.invert_yaxis()
    ax7.grid(True, alpha=0.3, axis='x')
else:
    ax7.text(0.5, 0.5, 'Нет данных с ненулевой СКП', ha='center', va='center', transform=ax7.transAxes)

# 8. Топ-10 IGS
ax8 = fig.add_subplot(3, 3, 8)
if len(df_igs_stats) > 0:
    top_igs = df_igs_stats[df_igs_stats['skp_3D_mm'] > 0].nsmallest(10, 'skp_3D_mm')
    if len(top_igs) > 0:
        y_pos = np.arange(len(top_igs))
        ax8.barh(y_pos, top_igs['skp_3D_mm'], color='red', edgecolor='black')
        ax8.set_yticks(y_pos)
        ax8.set_yticklabels(top_igs['station'], fontsize=8)
        ax8.set_xlabel('СКП_3D (мм)')
        ax8.set_title('Топ-10 точных станций IGS')
        ax8.invert_yaxis()
        ax8.grid(True, alpha=0.3, axis='x')
    else:
        ax8.text(0.5, 0.5, 'Нет данных с ненулевой СКП', ha='center', va='center', transform=ax8.transAxes)
else:
    ax8.text(0.5, 0.5, 'Нет данных для IGS', ha='center', va='center', transform=ax8.transAxes)
    ax8.set_title('Топ-10 точных станций IGS')

# 9. QQ-plot
ax9 = fig.add_subplot(3, 3, 9)
if len(fags_data) > 0:
    from scipy import stats as scipy_stats
    scipy_stats.probplot(fags_data, dist="norm", plot=ax9)
    ax9.get_lines()[0].set_color('blue')
    ax9.get_lines()[1].set_color('red')
    ax9.set_title('QQ-plot: СКП_3D (ФАГС)')
else:
    ax9.text(0.5, 0.5, 'Нет данных', ha='center', va='center', transform=ax9.transAxes)

plt.suptitle('СРАВНИТЕЛЬНЫЙ АНАЛИЗ ТОЧНОСТИ: ФАГС vs IGS', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fags_vs_igs_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================================================

df_fags_stats.to_csv('fags_stations_stats.csv', index=False)
print(f"\nСохранено: fags_stations_stats.csv ({len(df_fags_stats)} станций)")

if len(df_igs_stats) > 0:
    df_igs_stats.to_csv('igs_stations_stats.csv', index=False)
    print(f"Сохранено: igs_stations_stats.csv ({len(df_igs_stats)} станций)")

print("\n" + "=" * 80)
print("ИТОГОВОЕ РЕЗЮМЕ")
print("=" * 80)

print(f"\n📊 ФАГС: {summary_fags['count']} станций, средняя СКП_3D = {summary_fags['skp_3D_mean']:.2f} ± {summary_fags['skp_3D_std']:.2f} мм")
if summary_igs:
    print(f"📊 IGS: {summary_igs['count']} станций, средняя СКП_3D = {summary_igs['skp_3D_mean']:.2f} ± {summary_igs['skp_3D_std']:.2f} мм")
    
    diff = summary_fags['skp_3D_mean'] - summary_igs['skp_3D_mean']
    if diff < 0:
        print(f"\n✅ ФАГС точнее IGS на {abs(diff):.2f} мм")
    else:
        print(f"\n⚠️ IGS точнее ФАГС на {abs(diff):.2f} мм")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/Users/sergeidolin/anaconda3/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 711, in start
    self.io_loop.start()
  

AttributeError: _ARRAY_API not found

РАЗДЕЛЬНЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ ТОЧНОСТИ: ФАГС vs IGS

Загрузка данных ФАГС...
  - IGS данные загружены: 52000 записей

ФАГС: загружено 118 записей
IGS: загружено 52000 записей

ЭТАП 1: ФИЛЬТРАЦИЯ И РАСЧЕТ СТАТИСТИК
  - ФАГС: после фильтрации 72 измерений
  - ФАГС: после обработки 0 станций
  - IGS: после фильтрации 10133 измерений
  - IGS: после обработки 278 станций

ОШИБКА: Нет данных для ФАГС! Проверьте файл.

ЭТАП 2: СРАВНИТЕЛЬНАЯ СТАТИСТИКА ФАГС vs IGS

СРАВНИТЕЛЬНАЯ ТАБЛИЦА:
-----------------------------------------------------------------------------------------------
Параметр                            ФАГС                         IGS                         
-----------------------------------------------------------------------------------------------
Количество станций                  0                            278                         
Общее число измерений               0                            9777                        
                                      

: 

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Настройка стиля для научных публикаций
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['legend.fontsize'] = 10

print("=" * 80)
print("РАЗДЕЛЬНЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ ТОЧНОСТИ: ФАГС vs IGS")
print("=" * 80)

# ============================================================================
# ЗАГРУЗКА ДАННЫХ
# ============================================================================

# Загрузка статистики из FAGS файла
df_fags = pd.read_csv('/Users/sergeidolin/collaborative-service/fags_stations_statistics.csv')

# Создание DataFrame для IGS (на основе тех же данных, но с маркировкой)
# Примечание: в реальном анализе IGS данные должны быть загружены из отдельного файла
# Здесь мы создаем тестовые данные для демонстрации раздельного анализа

# Для демонстрации разделим станции по географическому принципу:
# - ФАГС: российские станции (по названию)
# - IGS: международные станции

def classify_network(station_name):
    """Классификация станции по сети ФАГС или IGS"""
    # Список российских станций (ФАГС)
    russian_stations = ['NSK1', 'ANDR', 'AST3', 'ANTP', 'AMDR', 'ARKH', 'BELG', 
                        'CHIT', 'AYN1', 'BORO', 'BARE', 'CNG1', 'CHRD', 'ELIS',
                        'IRKO', 'EKTG', 'FSVO', 'ITRP', 'IZVK', 'KAGP', 'KHAZ',
                        'KLCH', 'KIRV', 'KIZ1', 'KLN1', 'KNDL', 'LVR1', 'KLPS',
                        'KOTL', 'KZDV', 'MAG1', 'MBR1', 'LVR2', 'MHCH', 'MGNS',
                        'MBR2', 'MKR1', 'NERY', 'MURM', 'NARY', 'NIRO', 'NNOV',
                        'NOVG', 'OHA1', 'NOYA', 'OKTB', 'OLGA', 'OREN', 'OMSR',
                        'OXTK', 'ONGY', 'PEVK', 'PNGD', 'RGSK', 'RBCS', 'SEGE',
                        'SEMH', 'SPB2', 'SLH1', 'SVK1', 'TUAP', 'TILK', 'TIXG',
                        'TURR', 'TULN', 'UFGS', 'UNGN', 'VLS1', 'VLDV', 'VNOV',
                        'VRH1', 'ZVRG', 'ZHEL', 'VRH2']
    
    if station_name in russian_stations:
        return 'ФАГС'
    else:
        return 'IGS'

# Добавляем колонку с сетью
df_fags['network'] = df_fags['station'].apply(classify_network)

print(f"\nЗагружено данных: {len(df_fags)} станций")
print(f"  - ФАГС станций: {(df_fags['network'] == 'ФАГС').sum()}")
print(f"  - IGS станций: {(df_fags['network'] == 'IGS').sum()}")

# ============================================================================
# ЭТАП 1: ОЧИСТКА ДАННЫХ ОТ ВЫБРОСОВ
# ============================================================================
print("\n" + "=" * 80)
print("ЭТАП 1: ОЧИСТКА ДАННЫХ ОТ АНОМАЛЬНЫХ ИЗМЕРЕНИЙ")
print("=" * 80)

df_clean = df_fags.copy()

# Удаление станций с нулевой СКП
initial_count = len(df_clean)
df_clean = df_clean[df_clean['skp_3D_mm'] > 0]
print(f"  - Удалено станций с нулевой СКП: {initial_count - len(df_clean)}")

# Удаление станций с аномально высокой СКП (> 100 мм)
initial_count = len(df_clean)
df_clean = df_clean[df_clean['skp_3D_mm'] < 100]
print(f"  - Удалено станций с СКП_3D > 100 мм: {initial_count - len(df_clean)}")

# Минимальное количество измерений
initial_count = len(df_clean)
df_clean = df_clean[df_clean['n_meas'] >= 2]
print(f"  - Удалено станций с n_meas < 2: {initial_count - len(df_clean)}")

print(f"\nИтог после очистки: {len(df_clean)} станций")
print(f"  - ФАГС: {(df_clean['network'] == 'ФАГС').sum()}")
print(f"  - IGS: {(df_clean['network'] == 'IGS').sum()}")

# Разделение на два DataFrame
df_fags_clean = df_clean[df_clean['network'] == 'ФАГС'].copy()
df_igs_clean = df_clean[df_clean['network'] == 'IGS'].copy()

# ============================================================================
# ЭТАП 2: СРАВНИТЕЛЬНАЯ СТАТИСТИКА ФАГС vs IGS
# ============================================================================
print("\n" + "=" * 80)
print("ЭТАП 2: СРАВНИТЕЛЬНАЯ СТАТИСТИКА ФАГС vs IGS")
print("=" * 80)

def compute_network_stats(df, network_name):
    """Вычисление статистики для сети"""
    if len(df) == 0:
        return None
    
    stats_dict = {
        'network': network_name,
        'count': len(df),
        'skp_N_mean': df['skp_N_mm'].mean(),
        'skp_N_std': df['skp_N_mm'].std(),
        'skp_E_mean': df['skp_E_mm'].mean(),
        'skp_E_std': df['skp_E_mm'].std(),
        'skp_U_mean': df['skp_U_mm'].mean(),
        'skp_U_std': df['skp_U_mm'].std(),
        'skp_3D_mean': df['skp_3D_mm'].mean(),
        'skp_3D_median': df['skp_3D_mm'].median(),
        'skp_3D_std': df['skp_3D_mm'].std(),
        'bias_N_mean': df['bias_N_mm'].mean(),
        'bias_E_mean': df['bias_E_mm'].mean(),
        'bias_U_mean': df['bias_U_mm'].mean(),
        'bias_3D_mean': df['bias_3D_mm'].mean(),
        'mean_nsat_mean': df['mean_nsat'].mean(),
        'mean_nsat_std': df['mean_nsat'].std(),
        'n_meas_mean': df['n_meas'].mean(),
        'n_meas_total': df['n_meas'].sum()
    }
    
    # СКП по всем станциям (среднеквадратичное)
    stats_dict['skp_N_rms'] = np.sqrt(np.mean(df['skp_N_mm']**2))
    stats_dict['skp_E_rms'] = np.sqrt(np.mean(df['skp_E_mm']**2))
    stats_dict['skp_U_rms'] = np.sqrt(np.mean(df['skp_U_mm']**2))
    stats_dict['skp_3D_rms'] = np.sqrt(np.mean(df['skp_3D_mm']**2))
    
    return stats_dict

stats_fags = compute_network_stats(df_fags_clean, 'ФАГС')
stats_igs = compute_network_stats(df_igs_clean, 'IGS')

# Вывод сравнительной таблицы
print("\nСРАВНИТЕЛЬНАЯ ТАБЛИЦА СТАТИСТИК:")
print("-" * 100)
print(f"{'Параметр':<35} {'ФАГС':<25} {'IGS':<25}")
print("-" * 100)

comparison_data = [
    ('Количество станций', f"{stats_fags['count']}", f"{stats_igs['count']}"),
    ('Общее количество измерений', f"{stats_fags['n_meas_total']}", f"{stats_igs['n_meas_total']}"),
    ('', '', ''),
    ('СКП_N (среднее), мм', f"{stats_fags['skp_N_mean']:.2f} ± {stats_fags['skp_N_std']:.2f}", 
     f"{stats_igs['skp_N_mean']:.2f} ± {stats_igs['skp_N_std']:.2f}"),
    ('СКП_E (среднее), мм', f"{stats_fags['skp_E_mean']:.2f} ± {stats_fags['skp_E_std']:.2f}", 
     f"{stats_igs['skp_E_mean']:.2f} ± {stats_igs['skp_E_std']:.2f}"),
    ('СКП_U (среднее), мм', f"{stats_fags['skp_U_mean']:.2f} ± {stats_fags['skp_U_std']:.2f}", 
     f"{stats_igs['skp_U_mean']:.2f} ± {stats_igs['skp_U_std']:.2f}"),
    ('СКП_3D (среднее), мм', f"{stats_fags['skp_3D_mean']:.2f} ± {stats_fags['skp_3D_std']:.2f}", 
     f"{stats_igs['skp_3D_mean']:.2f} ± {stats_igs['skp_3D_std']:.2f}"),
    ('СКП_3D (медиана), мм', f"{stats_fags['skp_3D_median']:.2f}", f"{stats_igs['skp_3D_median']:.2f}"),
    ('', '', ''),
    ('СКП_N (СКО), мм', f"{stats_fags['skp_N_rms']:.2f}", f"{stats_igs['skp_N_rms']:.2f}"),
    ('СКП_E (СКО), мм', f"{stats_fags['skp_E_rms']:.2f}", f"{stats_igs['skp_E_rms']:.2f}"),
    ('СКП_U (СКО), мм', f"{stats_fags['skp_U_rms']:.2f}", f"{stats_igs['skp_U_rms']:.2f}"),
    ('СКП_3D (СКО), мм', f"{stats_fags['skp_3D_rms']:.2f}", f"{stats_igs['skp_3D_rms']:.2f}"),
    ('', '', ''),
    ('Сист. сдвиг dN, мм', f"{stats_fags['bias_N_mean']:.2f}", f"{stats_igs['bias_N_mean']:.2f}"),
    ('Сист. сдвиг dE, мм', f"{stats_fags['bias_E_mean']:.2f}", f"{stats_igs['bias_E_mean']:.2f}"),
    ('Сист. сдвиг dU, мм', f"{stats_fags['bias_U_mean']:.2f}", f"{stats_igs['bias_U_mean']:.2f}"),
    ('Сист. сдвиг 3D, мм', f"{stats_fags['bias_3D_mean']:.2f}", f"{stats_igs['bias_3D_mean']:.2f}"),
    ('', '', ''),
    ('Среднее NSAT', f"{stats_fags['mean_nsat_mean']:.1f} ± {stats_fags['mean_nsat_std']:.1f}", 
     f"{stats_igs['mean_nsat_mean']:.1f} ± {stats_igs['mean_nsat_std']:.1f}"),
    ('Среднее N измерений', f"{stats_fags['n_meas_mean']:.1f}", f"{stats_igs['n_meas_mean']:.1f}"),
]

for param, val_fags, val_igs in comparison_data:
    if param == '':
        print("")
    else:
        print(f"{param:<35} {val_fags:<25} {val_igs:<25}")

# ============================================================================
# ЭТАП 3: КАТЕГОРИЗАЦИЯ ПО ТОЧНОСТИ
# ============================================================================

def get_precision_category(skp_mm):
    """Категоризация станции по СКП_3D"""
    if skp_mm < 5:
        return "Экстра-прецизионные (<5 мм)"
    elif skp_mm < 10:
        return "Прецизионные (5-10 мм)"
    elif skp_mm < 20:
        return "Высокоточные (10-20 мм)"
    elif skp_mm < 40:
        return "Средней точности (20-40 мм)"
    else:
        return "Пониженной точности (>40 мм)"

df_fags_clean['category'] = df_fags_clean['skp_3D_mm'].apply(get_precision_category)
df_igs_clean['category'] = df_igs_clean['skp_3D_mm'].apply(get_precision_category)

print("\n" + "=" * 80)
print("ЭТАП 3: РАСПРЕДЕЛЕНИЕ ПО КАТЕГОРИЯМ ТОЧНОСТИ")
print("=" * 80)

# Статистика по категориям для ФАГС
category_fags = df_fags_clean.groupby('category').agg({
    'station': 'count',
    'skp_3D_mm': 'mean'
}).round(2)
category_fags.columns = ['ФАГС_кол-во', 'ФАГС_СКП_ср']

# Статистика по категориям для IGS
category_igs = df_igs_clean.groupby('category').agg({
    'station': 'count',
    'skp_3D_mm': 'mean'
}).round(2)
category_igs.columns = ['IGS_кол-во', 'IGS_СКП_ср']

# Объединение
category_compare = pd.concat([category_fags, category_igs], axis=1).fillna(0)
category_compare['Всего'] = category_compare['ФАГС_кол-во'] + category_compare['IGS_кол-во']
print("\nРаспределение станций по категориям точности:")
print(category_compare.to_string())

# ============================================================================
# ЭТАП 4: СТАТИСТИЧЕСКАЯ ПРОВЕРКА ГИПОТЕЗ
# ============================================================================
print("\n" + "=" * 80)
print("ЭТАП 4: СТАТИСТИЧЕСКАЯ ПРОВЕРКА РАЗЛИЧИЙ (ФАГС vs IGS)")
print("=" * 80)

from scipy.stats import mannwhitneyu, ttest_ind

def statistical_test(data1, data2, param_name):
    """Проведение статистических тестов"""
    # Тест Манна-Уитни (непараметрический)
    u_stat, p_value_mw = mannwhitneyu(data1, data2, alternative='two-sided')
    
    # t-тест (параметрический)
    t_stat, p_value_ttest = ttest_ind(data1, data2)
    
    return {
        'param': param_name,
        'median_1': np.median(data1),
        'median_2': np.median(data2),
        'mean_1': np.mean(data1),
        'mean_2': np.mean(data2),
        'p_value_mw': p_value_mw,
        'p_value_ttest': p_value_ttest,
        'significant': p_value_mw < 0.05
    }

params_to_test = [
    ('СКП_3D (мм)', df_fags_clean['skp_3D_mm'], df_igs_clean['skp_3D_mm']),
    ('СКП_N (мм)', df_fags_clean['skp_N_mm'], df_igs_clean['skp_N_mm']),
    ('СКП_E (мм)', df_fags_clean['skp_E_mm'], df_igs_clean['skp_E_mm']),
    ('СКП_U (мм)', df_fags_clean['skp_U_mm'], df_igs_clean['skp_U_mm']),
    ('NSAT', df_fags_clean['mean_nsat'], df_igs_clean['mean_nsat']),
]

test_results = []
for param_name, data1, data2 in params_to_test:
    if len(data1) > 0 and len(data2) > 0:
        result = statistical_test(data1, data2, param_name)
        test_results.append(result)

print("\nРезультаты статистических тестов (Манн-Уитни):")
print("-" * 90)
print(f"{'Параметр':<25} {'Медиана ФАГС':<15} {'Медиана IGS':<15} {'p-value':<12} {'Различие'}")
print("-" * 90)

for res in test_results:
    signif = "ДА ✓" if res['significant'] else "НЕТ ✗"
    print(f"{res['param']:<25} {res['median_1']:<15.2f} {res['median_2']:<15.2f} {res['p_value_mw']:<12.4f} {signif}")

# ============================================================================
# ПОСТРОЕНИЕ ГРАФИКОВ
# ============================================================================

fig = plt.figure(figsize=(18, 16))

# 1. Сравнение распределений СКП_3D (гистограммы)
ax1 = fig.add_subplot(3, 3, 1)
ax1.hist(df_fags_clean['skp_3D_mm'], bins=15, alpha=0.5, label='ФАГС', color='blue', edgecolor='black')
ax1.hist(df_igs_clean['skp_3D_mm'], bins=15, alpha=0.5, label='IGS', color='red', edgecolor='black')
ax1.set_xlabel('СКП_3D (мм)')
ax1.set_ylabel('Количество станций')
ax1.set_title('Распределение СКП_3D: ФАГС vs IGS')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Boxplot сравнение СКП по компонентам
ax2 = fig.add_subplot(3, 3, 2)
components = ['СКП_N', 'СКП_E', 'СКП_U']
x = np.arange(len(components))
width = 0.35

fags_means = [df_fags_clean['skp_N_mm'].mean(), df_fags_clean['skp_E_mm'].mean(), df_fags_clean['skp_U_mm'].mean()]
igs_means = [df_igs_clean['skp_N_mm'].mean(), df_igs_clean['skp_E_mm'].mean(), df_igs_clean['skp_U_mm'].mean()]
fags_stds = [df_fags_clean['skp_N_mm'].std(), df_fags_clean['skp_E_mm'].std(), df_fags_clean['skp_U_mm'].std()]
igs_stds = [df_igs_clean['skp_N_mm'].std(), df_igs_clean['skp_E_mm'].std(), df_igs_clean['skp_U_mm'].std()]

ax2.bar(x - width/2, fags_means, width, yerr=fags_stds, label='ФАГС', color='blue', capsize=3)
ax2.bar(x + width/2, igs_means, width, yerr=igs_stds, label='IGS', color='red', capsize=3)
ax2.set_xticks(x)
ax2.set_xticklabels(components)
ax2.set_ylabel('СКП (мм)')
ax2.set_title('Сравнение СКП по компонентам')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 3. Круговые диаграммы категорий точности
ax3 = fig.add_subplot(3, 3, 3)
categories = ['Экстра-прецизионные', 'Прецизионные', 'Высокоточные', 'Средней точности', 'Пониженной точности']
fags_counts = [len(df_fags_clean[df_fags_clean['category'] == cat]) for cat in categories]
igs_counts = [len(df_igs_clean[df_igs_clean['category'] == cat]) for cat in categories]

x = np.arange(len(categories))
width = 0.35
ax3.bar(x - width/2, fags_counts, width, label='ФАГС', color='blue')
ax3.bar(x + width/2, igs_counts, width, label='IGS', color='red')
ax3.set_xticks(x)
ax3.set_xticklabels(categories, rotation=45, ha='right', fontsize=8)
ax3.set_ylabel('Количество станций')
ax3.set_title('Распределение по категориям точности')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# 4. Сравнение систематических ошибок
ax4 = fig.add_subplot(3, 3, 4)
bias_components = ['dN', 'dE', 'dU']
x = np.arange(len(bias_components))

fags_biases = [df_fags_clean['bias_N_mm'].mean(), df_fags_clean['bias_E_mm'].mean(), df_fags_clean['bias_U_mm'].mean()]
igs_biases = [df_igs_clean['bias_N_mm'].mean(), df_igs_clean['bias_E_mm'].mean(), df_igs_clean['bias_U_mm'].mean()]

ax4.axhline(0, color='black', linestyle='-', linewidth=0.8)
ax4.bar(x - width/2, fags_biases, width, label='ФАГС', color='blue')
ax4.bar(x + width/2, igs_biases, width, label='IGS', color='red')
ax4.set_xticks(x)
ax4.set_xticklabels(bias_components)
ax4.set_ylabel('Систематическая ошибка (мм)')
ax4.set_title('Сравнение систематических ошибок')
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 5. Зависимость СКП_3D от NSAT
ax5 = fig.add_subplot(3, 3, 5)
ax5.scatter(df_fags_clean['mean_nsat'], df_fags_clean['skp_3D_mm'], 
            alpha=0.6, label='ФАГС', color='blue', s=40)
ax5.scatter(df_igs_clean['mean_nsat'], df_igs_clean['skp_3D_mm'], 
            alpha=0.6, label='IGS', color='red', s=40)
ax5.set_xlabel('Среднее количество спутников')
ax5.set_ylabel('СКП_3D (мм)')
ax5.set_title('Зависимость СКП_3D от NSAT')
ax5.legend()
ax5.grid(True, alpha=0.3)

# 6. QQ-plot для проверки нормальности распределения
ax6 = fig.add_subplot(3, 3, 6)
from scipy import stats as scipy_stats
scipy_stats.probplot(df_fags_clean['skp_3D_mm'], dist="norm", plot=ax6)
ax6.get_lines()[0].set_color('blue')
ax6.get_lines()[1].set_color('red')
ax6.set_title('QQ-plot: Распределение СКП_3D (ФАГС)')

# 7. Топ-10 лучших станций ФАГС
ax7 = fig.add_subplot(3, 3, 7)
top_fags = df_fags_clean.nsmallest(10, 'skp_3D_mm')
y_pos = np.arange(len(top_fags))
ax7.barh(y_pos, top_fags['skp_3D_mm'], color='blue', edgecolor='black')
ax7.set_yticks(y_pos)
ax7.set_yticklabels(top_fags['station'], fontsize=8)
ax7.set_xlabel('СКП_3D (мм)')
ax7.set_title('Топ-10 наиболее точных станций ФАГС')
ax7.invert_yaxis()
ax7.grid(True, alpha=0.3, axis='x')

# 8. Топ-10 лучших станций IGS
ax8 = fig.add_subplot(3, 3, 8)
top_igs = df_igs_clean.nsmallest(10, 'skp_3D_mm')
y_pos = np.arange(len(top_igs))
ax8.barh(y_pos, top_igs['skp_3D_mm'], color='red', edgecolor='black')
ax8.set_yticks(y_pos)
ax8.set_yticklabels(top_igs['station'], fontsize=8)
ax8.set_xlabel('СКП_3D (мм)')
ax8.set_title('Топ-10 наиболее точных станций IGS')
ax8.invert_yaxis()
ax8.grid(True, alpha=0.3, axis='x')

# 9. Violin plot для сравнения распределений
ax9 = fig.add_subplot(3, 3, 9)
data_to_plot = [df_fags_clean['skp_3D_mm'], df_igs_clean['skp_3D_mm']]
vp = ax9.violinplot(data_to_plot, positions=[1, 2], showmeans=True, showmedians=True)
vp['bodies'][0].set_facecolor('blue')
vp['bodies'][0].set_alpha(0.6)
vp['bodies'][1].set_facecolor('red')
vp['bodies'][1].set_alpha(0.6)
ax9.set_xticks([1, 2])
ax9.set_xticklabels(['ФАГС', 'IGS'])
ax9.set_ylabel('СКП_3D (мм)')
ax9.set_title('Сравнение распределений СКП_3D')
ax9.grid(True, alpha=0.3, axis='y')

plt.suptitle('СРАВНИТЕЛЬНЫЙ АНАЛИЗ ТОЧНОСТИ GNSS-СТАНЦИЙ: ФАГС vs IGS',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('fags_vs_igs_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================================
# ДОПОЛНИТЕЛЬНЫЙ ГРАФИК: Корреляционные матрицы
# ============================================================================

fig2, axes = plt.subplots(1, 2, figsize=(14, 6))

# Корреляционная матрица для ФАГС
corr_fags = df_fags_clean[['skp_N_mm', 'skp_E_mm', 'skp_U_mm', 'skp_3D_mm', 
                            'mean_nsat', 'n_meas']].copy()
corr_fags.columns = ['СКП_N', 'СКП_E', 'СКП_U', 'СКП_3D', 'NSAT', 'N_изм']
sns.heatmap(corr_fags.corr(), annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=axes[0])
axes[0].set_title('Корреляционная матрица (ФАГС)', fontsize=12, fontweight='bold')

# Корреляционная матрица для IGS
corr_igs = df_igs_clean[['skp_N_mm', 'skp_E_mm', 'skp_U_mm', 'skp_3D_mm', 
                          'mean_nsat', 'n_meas']].copy()
corr_igs.columns = ['СКП_N', 'СКП_E', 'СКП_U', 'СКП_3D', 'NSAT', 'N_изм']
sns.heatmap(corr_igs.corr(), annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, ax=axes[1])
axes[1].set_title('Корреляционная матрица (IGS)', fontsize=12, fontweight='bold')

plt.suptitle('Сравнение корреляционных матриц: ФАГС vs IGS', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fags_vs_igs_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# ============================================================================
# СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================================================

# Сохранение очищенных данных
df_fags_clean.to_csv('fags_network_statistics.csv', index=False)
df_igs_clean.to_csv('igs_network_statistics.csv', index=False)

# Сохранение сравнительной статистики
comparison_df = pd.DataFrame(comparison_data[1:], columns=['Параметр', 'ФАГС', 'IGS'])
comparison_df.to_csv('fags_vs_igs_comparison.csv', index=False)

print(f"\nРезультаты сохранены в файлы:")
print(f"  - fags_network_statistics.csv (ФАГС)")
print(f"  - igs_network_statistics.csv (IGS)")
print(f"  - fags_vs_igs_comparison.csv (сравнение)")

# ============================================================================
# ИТОГОВОЕ РЕЗЮМЕ
# ============================================================================

print("\n" + "=" * 80)
print("ИТОГОВОЕ РЕЗЮМЕ СРАВНИТЕЛЬНОГО АНАЛИЗА")
print("=" * 80)

print(f"\nФАГС (Российская сеть):")
print(f"  • Всего станций: {stats_fags['count']}")
print(f"  • Средняя СКП_3D: {stats_fags['skp_3D_mean']:.2f} ± {stats_fags['skp_3D_std']:.2f} мм")
print(f"  • Медианная СКП_3D: {stats_fags['skp_3D_median']:.2f} мм")
print(f"  • Общая СКП_3D (СКО): {stats_fags['skp_3D_rms']:.2f} мм")
print(f"  • Среднее NSAT: {stats_fags['mean_nsat_mean']:.1f}")

print(f"\nIGS (Международная сеть):")
print(f"  • Всего станций: {stats_igs['count']}")
print(f"  • Средняя СКП_3D: {stats_igs['skp_3D_mean']:.2f} ± {stats_igs['skp_3D_std']:.2f} мм")
print(f"  • Медианная СКП_3D: {stats_igs['skp_3D_median']:.2f} мм")
print(f"  • Общая СКП_3D (СКО): {stats_igs['skp_3D_rms']:.2f} мм")
print(f"  • Среднее NSAT: {stats_igs['mean_nsat_mean']:.1f}")

# Сравнение
diff_3d = stats_fags['skp_3D_mean'] - stats_igs['skp_3D_mean']
if diff_3d > 0:
    print(f"\n📊 Вывод: Сеть ФАГС показывает {'худшую' if diff_3d > 0 else 'лучшую'} точность по сравнению с IGS")
    print(f"   Разница в средней СКП_3D составляет {abs(diff_3d):.2f} мм")
else:
    print(f"\n📊 Вывод: Сеть ФАГС показывает лучшую точность по сравнению с IGS")
    print(f"   Разница в средней СКП_3D составляет {abs(diff_3d):.2f} мм")

РАЗДЕЛЬНЫЙ СТАТИСТИЧЕСКИЙ АНАЛИЗ ТОЧНОСТИ: ФАГС vs IGS

Загружено данных: 1 станций
  - ФАГС станций: 1
  - IGS станций: 0

ЭТАП 1: ОЧИСТКА ДАННЫХ ОТ АНОМАЛЬНЫХ ИЗМЕРЕНИЙ
  - Удалено станций с нулевой СКП: 1
  - Удалено станций с СКП_3D > 100 мм: 0
  - Удалено станций с n_meas < 2: 0

Итог после очистки: 0 станций
  - ФАГС: 0
  - IGS: 0

ЭТАП 2: СРАВНИТЕЛЬНАЯ СТАТИСТИКА ФАГС vs IGS

СРАВНИТЕЛЬНАЯ ТАБЛИЦА СТАТИСТИК:
----------------------------------------------------------------------------------------------------
Параметр                            ФАГС                      IGS                      
----------------------------------------------------------------------------------------------------


TypeError: 'NoneType' object is not subscriptable